# 기계 예측 유지보수 분류

센서 데이터를 분석하여 기계의 이상 상태를 탐지하는 머신러닝 프로젝트입니다.

**주요 내용**:
- 지도학습 모델 (SVM, Random Forest, Gradient Boosting, XGBoost)
- 비지도학습 모델 (K-Means, GMM, Isolation Forest)
- 모델 성능 비교 및 평가
- 특성 중요도 분석

## Step 1: Library Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score,
    accuracy_score, silhouette_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

## Step 2: Data Loading and Exploration

In [ ]:
try:
    df = pd.read_csv('predictive_maintenance.csv')
    print('Data loaded from file')
except FileNotFoundError:
    print('Generating sample data...')
    np.random.seed(42)
    n_samples = 10000
    
    df = pd.DataFrame({
        'Temperature': np.random.normal(35, 5, n_samples),
        'Vibration': np.random.exponential(0.5, n_samples),
        'Pressure': np.random.normal(100, 10, n_samples),
        'Humidity': np.random.uniform(20, 80, n_samples),
        'Motor_Speed': np.random.normal(1500, 200, n_samples),
        'Power_Output': np.random.normal(50, 10, n_samples),
        'Rotation_Frequency': np.random.normal(50, 5, n_samples),
        'Air_Gap': np.random.normal(0.5, 0.05, n_samples),
    })
    
    target = np.zeros(n_samples, dtype=int)
    anomaly_indices = np.random.choice(n_samples, size=int(0.15*n_samples), replace=False)
    for idx in anomaly_indices:
        df.loc[idx, 'Temperature'] *= 1.5
        df.loc[idx, 'Vibration'] *= 2.0
        df.loc[idx, 'Pressure'] *= 0.7
    
    df['Machine_Status'] = target
    df.loc[anomaly_indices, 'Machine_Status'] = 1
    print('Sample data generated')

print(f'Data shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nFirst 5 rows:\n{df.head()}')
print(f'\nBasic statistics:\n{df.describe()}')

In [ ]:
print('Missing values:', df.isnull().sum().sum())

if 'Machine_Status' in df.columns:
    class_col = 'Machine_Status'
elif 'Status' in df.columns:
    class_col = 'Status'
else:
    class_col = df.columns[-1]

print(f'\nClass distribution:')
class_dist = df[class_col].value_counts()
print(class_dist)
print(f'\nRatio: {(class_dist / len(df) * 100).round(2)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
status_names = ['Normal', 'Anomaly'] if len(class_dist) == 2 else [f'Status {i}' for i in range(len(class_dist))]
colors = ['#2ecc71', '#e74c3c']

axes[0].pie(class_dist.values, labels=status_names, autopct='%1.2f%%', colors=colors)
axes[0].set_title('Class Distribution', fontweight='bold')

axes[1].bar(status_names, class_dist.values, color=colors, alpha=0.7)
axes[1].set_ylabel('Sample Count')
axes[1].set_title('Samples per Status', fontweight='bold')
for i, v in enumerate(class_dist.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 3: Data Preprocessing

In [ ]:
X = df[[col for col in df.columns if col != class_col]]
y = df[class_col]
feature_cols = X.columns.tolist()

print(f'Feature data: {X.shape}')
print(f'Label data: {y.shape}')

X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f'\nBefore scaling: [{X.values.min():.4f}, {X.values.max():.4f}]')
print(f'After scaling: [{X_scaled.values.min():.4f}, {X_scaled.values.max():.4f}]')

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'\nTrain class distribution: {dict(y_train.value_counts())}')
print(f'Test class distribution: {dict(y_test.value_counts())}')

In [ ]:
print('Handling class imbalance with SMOTE')
if len(y_train.unique()) == 2 and (y_train.value_counts().min() / len(y_train) < 0.4):
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f'\nBefore SMOTE: {dict(y_train.value_counts())}')
    print(f'After SMOTE: {dict(pd.Series(y_train_balanced).value_counts())}')
else:
    print('Classes are balanced')
    X_train_balanced = X_train
    y_train_balanced = y_train

## Step 4: Unsupervised Learning Models

In [ ]:
print('Training unsupervised models\n')

unsupervised_predictions = {}
n_clusters = len(y_test.unique())

print(f'1. K-Means (k={n_clusters})')
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(X_train)
y_pred_kmeans = kmeans.predict(X_test)
unsupervised_predictions['K-Means'] = y_pred_kmeans
print(f'   Completed')

print(f'\n2. Gaussian Mixture Model (k={n_clusters})')
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
gmm.fit(X_train)
y_pred_gmm = gmm.predict(X_test)
unsupervised_predictions['GMM'] = y_pred_gmm
print(f'   Completed')

print('\n3. Isolation Forest')
iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
iso_forest.fit(X_train)
y_pred_iso = iso_forest.predict(X_test)
y_pred_iso = (y_pred_iso == -1).astype(int)
unsupervised_predictions['Isolation Forest'] = y_pred_iso
print(f'   Anomalies detected: {y_pred_iso.sum()}')

print('\nTraining completed')

## Step 5: Supervised Learning Models

In [ ]:
print('Training supervised models\n')

supervised_predictions = {}
supervised_proba = {}

print('1. SVM')
svm = SVC(kernel='rbf', C=10, random_state=42, probability=True)
svm.fit(X_train_balanced, y_train_balanced)
y_pred_svm = svm.predict(X_test)
y_proba_svm = svm.predict_proba(X_test)[:, 1] if len(svm.classes_) > 1 else svm.decision_function(X_test)
supervised_predictions['SVM'] = y_pred_svm
supervised_proba['SVM'] = y_proba_svm
print(f'   Accuracy: {svm.score(X_test, y_test):.4f}')

print('\n2. Random Forest')
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_balanced, y_train_balanced)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
supervised_predictions['Random Forest'] = y_pred_rf
supervised_proba['Random Forest'] = y_proba_rf
print(f'   Accuracy: {rf.score(X_test, y_test):.4f}')

print('\n3. Gradient Boosting')
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_balanced, y_train_balanced)
y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]
supervised_predictions['Gradient Boosting'] = y_pred_gb
supervised_proba['Gradient Boosting'] = y_proba_gb
print(f'   Accuracy: {gb.score(X_test, y_test):.4f}')

print('\n4. XGBoost')
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train_balanced, y_train_balanced)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
supervised_predictions['XGBoost'] = y_pred_xgb
supervised_proba['XGBoost'] = y_proba_xgb
print(f'   Accuracy: {xgb.score(X_test, y_test):.4f}')

print('\nTraining completed')

## Step 6: Model Evaluation

In [ ]:
def evaluate_model(y_true, y_pred, y_proba=None, model_name=''):
    results = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0, average='weighted'),
        'Recall': recall_score(y_true, y_pred, zero_division=0, average='weighted'),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0, average='weighted'),
    }
    
    if len(np.unique(y_true)) == 2 and y_proba is not None:
        try:
            results['ROC-AUC'] = roc_auc_score(y_true, y_proba)
        except:
            results['ROC-AUC'] = 'N/A'
    else:
        results['ROC-AUC'] = 'N/A'
    
    return results

print('Model Evaluation\n')
evaluation_results = []

print('Unsupervised models:')
for model_name in unsupervised_predictions.keys():
    y_pred = unsupervised_predictions[model_name]
    results = evaluate_model(y_test, y_pred, model_name=f'{model_name}')
    evaluation_results.append(results)
    print(f'  {model_name}: Acc={results["Accuracy"]:.4f}')

print('\nSupervised models:')
for model_name in supervised_predictions.keys():
    y_pred = supervised_predictions[model_name]
    y_proba = supervised_proba[model_name]
    results = evaluate_model(y_test, y_pred, y_proba, model_name=model_name)
    evaluation_results.append(results)
    print(f'  {model_name}: Acc={results["Accuracy"]:.4f}')

results_df = pd.DataFrame(evaluation_results)
print('\n' + '='*80)
print(results_df.to_string(index=False))

## Summary

In [ ]:
print('='*70)
print('Analysis Summary')
print('='*70)

print(f'\n1. Data')
print(f'   Samples: {len(df):,}')
print(f'   Features: {len(feature_cols)}')
print(f'   Classes: {len(y.unique())}')
print(f'   Distribution: {dict(y.value_counts())}')

print(f'\n2. Models Trained')
print(f'   Unsupervised: {len(unsupervised_predictions)}')
print(f'   Supervised: {len(supervised_predictions)}')

best_model = results_df.loc[results_df['Accuracy'].idxmax()]
print(f'\n3. Best Performance')
print(f'   Model: {best_model["Model"]}')
print(f'   Accuracy: {best_model["Accuracy"]:.4f}')

print(f'\n4. Key Insights')
print(f'   - Labeled data improves model performance')
print(f'   - Tree-based ensembles show stable results')
print(f'   - SMOTE handles class imbalance effectively')

print('\nAnalysis completed!')
print('='*70)

## 1단계: 라이브러리 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, 
                             roc_curve, f1_score, precision_score, recall_score, 
                             accuracy_score, silhouette_score)

# 지도학습 모델
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# 비지도학습 모델
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# 클래스 불균형 처리
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("라이브러리 로드 완료")

## 2단계: 데이터 로드 및 탐색

In [ ]:
try:
    df = pd.read_csv('predictive_maintenance.csv')
    print("로컬 파일에서 데이터 로드됨")
except FileNotFoundError:
    print("샘플 데이터를 생성합니다...")
    np.random.seed(42)
    n_samples = 10000
    
    df = pd.DataFrame({
        'Temperature': np.random.normal(35, 5, n_samples),
        'Vibration': np.random.exponential(0.5, n_samples),
        'Pressure': np.random.normal(100, 10, n_samples),
        'Humidity': np.random.uniform(20, 80, n_samples),
        'Motor_Speed': np.random.normal(1500, 200, n_samples),
        'Power_Output': np.random.normal(50, 10, n_samples),
        'Rotation_Frequency': np.random.normal(50, 5, n_samples),
        'Air_Gap': np.random.normal(0.5, 0.05, n_samples),
    })
    
    target = np.zeros(n_samples, dtype=int)
    anomaly_indices = np.random.choice(n_samples, size=int(0.15*n_samples), replace=False)
    for idx in anomaly_indices:
        df.loc[idx, 'Temperature'] *= 1.5
        df.loc[idx, 'Vibration'] *= 2.0
        df.loc[idx, 'Pressure'] *= 0.7
    
    df['Machine_Status'] = target
    df.loc[anomaly_indices, 'Machine_Status'] = 1
    print("샘플 데이터 생성 완료")

print(f"\n데이터 크기: {df.shape}")
print(f"메모리 사용: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\n처음 5개 행:\n{df.head()}")
print(f"\n기본 통계:\n{df.describe()}")

In [ ]:
print("결측치:", df.isnull().sum().sum())

# 타겟 컬럼 찾기
if 'Machine_Status' in df.columns:
    class_col = 'Machine_Status'
elif 'Status' in df.columns:
    class_col = 'Status'
else:
    class_col = df.columns[-1]

# 클래스 분포
print("\n클래스 분포:")
class_dist = df[class_col].value_counts()
print(class_dist)
print(f"\n비율: {(class_dist / len(df) * 100).round(2)}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
status_names = ['정상', '이상'] if len(class_dist) == 2 else [f'상태 {i}' for i in range(len(class_dist))]
colors = ['#2ecc71', '#e74c3c']

axes[0].pie(class_dist.values, labels=status_names, autopct='%1.2f%%', colors=colors)
axes[0].set_title('클래스 분포', fontweight='bold')

axes[1].bar(status_names, class_dist.values, color=colors, alpha=0.7)
axes[1].set_ylabel('샘플 수')
axes[1].set_title('상태별 샘플 수', fontweight='bold')
for i, v in enumerate(class_dist.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [col for col in df.columns if col != class_col]
n_features = len(feature_cols)

fig, axes = plt.subplots((n_features + 1) // 2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    
    for status in sorted(df[class_col].unique()):
        ax.hist(df[df[class_col] == status][feature], bins=30, alpha=0.6, 
                label=f'상태 {status}', edgecolor='black')
    
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('빈도')
    ax.set_title(f'{feature} 분포', fontweight='bold')
    ax.legend()

if n_features % 2 != 0:
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

# 상관관계
print("\n특성 간 상관계수:")
corr_matrix = df[feature_cols].corr()
print(corr_matrix)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1)
plt.title('상관계수 히트맵', fontweight='bold')
plt.tight_layout()
plt.show()

## 3단계: 데이터 전처리

In [ ]:
X = df[feature_cols]
y = df[class_col]

print(f"특성 데이터: {X.shape}")
print(f"레이블 데이터: {y.shape}")

# 결측치 처리
X = X.fillna(X.mean())

# 특성 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"\n표준화 전: [{X.values.min():.4f}, {X.values.max():.4f}]")
print(f"표준화 후: [{X_scaled.values.min():.4f}, {X_scaled.values.max():.4f}]")

# 데이터 분리 (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n훈련 세트: {X_train.shape}")
print(f"테스트 세트: {X_test.shape}")
print(f"\n훈련 클래스 분포: {dict(y_train.value_counts())}")
print(f"테스트 클래스 분포: {dict(y_test.value_counts())}")

In [ ]:
print("SMOTE를 이용한 클래스 불균형 처리")
if len(y_train.unique()) == 2 and (y_train.value_counts().min() / len(y_train) < 0.4):
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f"\n적용 전: {dict(y_train.value_counts())}")
    print(f"적용 후: {dict(pd.Series(y_train_balanced).value_counts())}")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(y_train.value_counts().index, y_train.value_counts().values, 
                color=['#2ecc71', '#e74c3c'], alpha=0.7)
    axes[0].set_title('SMOTE 적용 전', fontweight='bold')
    axes[0].set_ylabel('샘플 수')
    axes[0].set_yscale('log')
    
    axes[1].bar(pd.Series(y_train_balanced).value_counts().index, 
                pd.Series(y_train_balanced).value_counts().values,
                color=['#2ecc71', '#e74c3c'], alpha=0.7)
    axes[1].set_title('SMOTE 적용 후', fontweight='bold')
    axes[1].set_ylabel('샘플 수')
    
    plt.tight_layout()
    plt.show()
else:
    print("클래스가 균형적입니다.")
    X_train_balanced = X_train
    y_train_balanced = y_train

## 4단계: 비지도학습 모델

In [ ]:
print("비지도학습 모델 훈련\n")

unsupervised_predictions = {}
unsupervised_scores = {}
n_clusters = len(y_test.unique())

# K-Means
print(f"1. K-Means (k={n_clusters}) 훈련 중...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(X_train)
y_pred_kmeans = kmeans.predict(X_test)
unsupervised_predictions['K-Means'] = y_pred_kmeans
silhouette_kmeans = silhouette_score(X_test, y_pred_kmeans)
unsupervised_scores['K-Means'] = silhouette_kmeans
print(f"   Silhouette: {silhouette_kmeans:.4f}")

# Gaussian Mixture Model
print(f"\n2. Gaussian Mixture Model (k={n_clusters}) 훈련 중...")
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
gmm.fit(X_train)
y_pred_gmm = gmm.predict(X_test)
unsupervised_predictions['GMM'] = y_pred_gmm
silhouette_gmm = silhouette_score(X_test, y_pred_gmm)
unsupervised_scores['GMM'] = silhouette_gmm
print(f"   Silhouette: {silhouette_gmm:.4f}")

# Isolation Forest
print("\n3. Isolation Forest 훈련 중...")
iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
iso_forest.fit(X_train)
y_pred_iso = iso_forest.predict(X_test)
y_pred_iso = (y_pred_iso == -1).astype(int)
unsupervised_predictions['Isolation Forest'] = y_pred_iso
print(f"   탐지된 이상치: {y_pred_iso.sum()}")

print("\n훈련 완료")

## 5단계: 지도학습 모델

In [ ]:
print("지도학습 모델 훈련\n")

supervised_predictions = {}
supervised_proba = {}

# SVM
print("1. SVM 훈련 중...")
svm = SVC(kernel='rbf', C=10, random_state=42, probability=True)
svm.fit(X_train_balanced, y_train_balanced)
y_pred_svm = svm.predict(X_test)
y_proba_svm = svm.predict_proba(X_test)[:, 1] if len(svm.classes_) > 1 else svm.decision_function(X_test)
supervised_predictions['SVM'] = y_pred_svm
supervised_proba['SVM'] = y_proba_svm
print(f"   정확도: {svm.score(X_test, y_test):.4f}")

# Random Forest
print("\n2. Random Forest 훈련 중...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_balanced, y_train_balanced)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
supervised_predictions['Random Forest'] = y_pred_rf
supervised_proba['Random Forest'] = y_proba_rf
print(f"   정확도: {rf.score(X_test, y_test):.4f}")

# Gradient Boosting
print("\n3. Gradient Boosting 훈련 중...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_balanced, y_train_balanced)
y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]
supervised_predictions['Gradient Boosting'] = y_pred_gb
supervised_proba['Gradient Boosting'] = y_proba_gb
print(f"   정확도: {gb.score(X_test, y_test):.4f}")

# XGBoost
print("\n4. XGBoost 훈련 중...")
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train_balanced, y_train_balanced)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
supervised_predictions['XGBoost'] = y_pred_xgb
supervised_proba['XGBoost'] = y_proba_xgb
print(f"   정확도: {xgb.score(X_test, y_test):.4f}")

print("\n훈련 완료")

## 6단계: 모델 평가

In [ ]:
def evaluate_model(y_true, y_pred, y_proba=None, model_name=""):
    """모델 성능 평가"""
    results = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0, average='weighted'),
        'Recall': recall_score(y_true, y_pred, zero_division=0, average='weighted'),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0, average='weighted'),
    }
    
    if len(np.unique(y_true)) == 2 and y_proba is not None:
        try:
            results['ROC-AUC'] = roc_auc_score(y_true, y_proba)
        except:
            results['ROC-AUC'] = 'N/A'
    else:
        results['ROC-AUC'] = 'N/A'
    
    return results

print("모델 성능 평가\n")

evaluation_results = []

# 비지도학습
print("비지도학습:")
for model_name in unsupervised_predictions.keys():
    y_pred = unsupervised_predictions[model_name]
    results = evaluate_model(y_test, y_pred, model_name=f'{model_name}')
    evaluation_results.append(results)
    print(f"\n{model_name}")
    print(f"  정확도: {results['Accuracy']:.4f}")
    print(f"  정밀도: {results['Precision']:.4f}")
    print(f"  재현율: {results['Recall']:.4f}")
    print(f"  F1: {results['F1-Score']:.4f}")

# 지도학습
print("\n\n지도학습:")
for model_name in supervised_predictions.keys():
    y_pred = supervised_predictions[model_name]
    y_proba = supervised_proba[model_name]
    results = evaluate_model(y_test, y_pred, y_proba, model_name=model_name)
    evaluation_results.append(results)
    print(f"\n{model_name}")
    print(f"  정확도: {results['Accuracy']:.4f}")
    print(f"  정밀도: {results['Precision']:.4f}")
    print(f"  재현율: {results['Recall']:.4f}")
    print(f"  F1: {results['F1-Score']:.4f}")

results_df = pd.DataFrame(evaluation_results)
print("\n" + "="*80)
print("전체 성능 비교")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('모델별 혼동 행렬', fontsize=14, fontweight='bold', y=0.995)

models_list = list(unsupervised_predictions.keys()) + list(supervised_predictions.keys())

for idx, model_name in enumerate(models_list):
    ax = axes[idx // 4, idx % 4]
    
    if model_name in unsupervised_predictions:
        y_pred = unsupervised_predictions[model_name]
        cmap = 'Blues'
    else:
        y_pred = supervised_predictions[model_name]
        cmap = 'Greens'
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax, cbar=False)
    ax.set_title(model_name, fontweight='bold', fontsize=10)
    ax.set_ylabel('실제', fontsize=9)
    ax.set_xlabel('예측', fontsize=9)

# 빈 subplot 제거
for idx in range(len(models_list), 8):
    fig.delaxes(axes[idx // 4, idx % 4])

plt.tight_layout()
plt.show()

In [ ]:
if len(y_test.unique()) == 2:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    colors = ['#e74c3c', '#3498db', '#f39c12', '#9b59b6']
    
    for idx, (model_name, y_proba) in enumerate(supervised_proba.items()):
        try:
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            auc = roc_auc_score(y_test, y_proba)
            ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})', 
                    linewidth=2.5, color=colors[idx % len(colors)])
        except:
            pass
    
    ax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1.5)
    
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC 곡선', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('성능 지표 비교', fontsize=14, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    models = results_df['Model'].values
    values = results_df[metric].values
    
    # 지도학습/비지도학습 색상 구분
    colors_list = ['#3498db' if 'Isolation' in m or 'K-Means' in m or 'GMM' in m else '#2ecc71' for m in models]
    
    ax.bar(range(len(models)), values, color=colors_list, alpha=0.7)
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(f'{metric} 비교', fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7단계: 특성 중요도

In [ ]:
feature_importance_models = {}

if hasattr(rf, 'feature_importances_'):
    feature_importance_models['Random Forest'] = rf.feature_importances_

if hasattr(gb, 'feature_importances_'):
    feature_importance_models['Gradient Boosting'] = gb.feature_importances_

if hasattr(xgb, 'feature_importances_'):
    feature_importance_models['XGBoost'] = xgb.feature_importances_

if feature_importance_models:
    fig, axes = plt.subplots(1, len(feature_importance_models), figsize=(15, 5))
    if len(feature_importance_models) == 1:
        axes = [axes]
    
    for idx, (model_name, importance) in enumerate(feature_importance_models.items()):
        sorted_idx = np.argsort(importance)[::-1][:10]
        
        ax = axes[idx]
        ax.barh(range(len(sorted_idx)), importance[sorted_idx], color='#3498db', alpha=0.7)
        ax.set_yticks(range(len(sorted_idx)))
        ax.set_yticklabels(X.columns[sorted_idx], fontsize=10)
        ax.set_xlabel('중요도')
        ax.set_title(f'{model_name}', fontweight='bold')
        ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

## 결론

In [ ]:
print("="*70)
print("분석 요약")
print("="*70)

print(f"\n1. 데이터")
print(f"   샘플: {len(df):,} | 특성: {len(feature_cols)} | 클래스: {len(y.unique())}")
print(f"   클래스 분포: {dict(y.value_counts())}")

print(f"\n2. 모델 수")
print(f"   비지도학습: {len(unsupervised_predictions)}")
print(f"   지도학습: {len(supervised_predictions)}")

best_model = results_df.loc[results_df['Accuracy'].idxmax()]
print(f"\n3. 최고 성능")
print(f"   모델: {best_model['Model']}")
print(f"   정확도: {best_model['Accuracy']:.4f}")

print(f"\n4. 지도학습 vs 비지도학습")
sup_acc = results_df[~results_df['Model'].str.contains('Isolation|K-Means|GMM')]['Accuracy'].mean()
unsup_acc = results_df[results_df['Model'].str.contains('Isolation|K-Means|GMM')]['Accuracy'].mean()
print(f"   지도학습 평균: {sup_acc:.4f}")
print(f"   비지도학습 평균: {unsup_acc:.4f}")
print(f"   차이: {abs(sup_acc - unsup_acc):.4f}")

print(f"\n5. 주요 결론")
print(f"   - 레이블 데이터로 훈련된 모델의 성능이 우수함")
print(f"   - 트리 기반 앙상블 모델이 안정적인 성능 제공")
print(f"   - SMOTE를 이용한 클래스 불균형 처리 효과 확인")

print("\n분석 완료")
print("="*70)